# Regression Method 1 — WPPSI-III IQ from Neonatal ECG/HRV

Template for regressing WPPSI-III IQ scores (QIT Full-Scale, QIV Verbal, QIP Performance, …) from neonatal linear + nonlinear HRV features (AuBE cohort, ~189 children, M0 recordings only).

Set ``TARGET_COLUMN`` below to the IQ you want to predict. To compare all IQs at once, use ``model_evaluation.py`` at the project root.

**Research only — not a diagnostic tool.** Health data lives in `data/` and must **never** be committed or shared with AI tools.

See `Group A/dataset.md` for the full data dictionary and merge rules.

In [ ]:
# --------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Persistence
import joblib

# Plotting
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Load Data

Load the three CSVs from `data/` following the rules in `dataset.md`:
- HRV files: skip row 2 (units row) → `skiprows=[1]`
- WPPSI-III file: second row is the header → `header=1`, tab-separated → `sep="\t"`
- Fix comma decimal separators in WPPSI-III (e.g. `99,6` → `99.6`)

In [ ]:
# --------------------------------------------------------------
# Paths (relative to project root)
# --------------------------------------------------------------

LINEAR_PATH = "data/HRV_Lineaire_Full.csv"
NONLINEAR_PATH = "data/HRV_NonLineaire_Full.csv"
WPPSI_PATH = "data/Résultats WPPSI III.csv"


# --------------------------------------------------------------
# 2a. Load linear HRV (skip units row at line 2)
# --------------------------------------------------------------

df_linear = pd.read_csv(LINEAR_PATH, skiprows=[1])
print("Linear HRV shape:", df_linear.shape)


# --------------------------------------------------------------
# 2b. Load nonlinear HRV (skip units row at line 2)
# --------------------------------------------------------------

df_nonlinear = pd.read_csv(NONLINEAR_PATH, skiprows=[1])
print("Nonlinear HRV shape:", df_nonlinear.shape)


# --------------------------------------------------------------
# 2c. Load WPPSI-III (header on row 2, tab-sep, comma decimals)
# --------------------------------------------------------------

df_wppsi = pd.read_csv(WPPSI_PATH, header=1, sep="\t")

# Fix comma decimal separator -> dot, then cast to numeric
df_wppsi = df_wppsi.replace({",": "."}, regex=True)
for col in df_wppsi.columns:
    df_wppsi[col] = pd.to_numeric(df_wppsi[col], errors="ignore")

print("WPPSI-III shape:", df_wppsi.shape)
print(df_wppsi.head())

## 3. Data Exploration

Inspect the raw frames: shapes, key identifier columns, Passage values, and missingness.

In [ ]:
print("Linear columns:", list(df_linear.columns))
print("\nNonlinear columns:", list(df_nonlinear.columns))
print("\nWPPSI columns:", list(df_wppsi.columns))

print("\nPassage values (linear):", df_linear["Passage"].unique() if "Passage" in df_linear.columns else "N/A")
print("Passage values (nonlinear):", df_nonlinear["Passage"].unique() if "Passage" in df_nonlinear.columns else "N/A")

print("\nMissing values per IQ column:")
for col in ["QIT", "QIV", "QIP", "NCL"]:
    if col in df_wppsi.columns:
        print(f"  {col}: {df_wppsi[col].isna().sum()} missing of {df_wppsi[col].notna().sum() + df_wppsi[col].isna().sum()}")

## 4. Preprocessing & Merge

Following `dataset.md` §7:
1. Choose a target IQ column (QIT, QIV, QIP, …)
2. Keep only `Passage == "M0"` (birth recordings)
3. Drop rows missing the chosen target
4. Merge linear + nonlinear on `Monogramme` + `DDN` + `Passage`
5. Merge HRV + WPPSI on `Monogramme` + `DDN`
6. Drop fully-empty feature columns
7. Impute remaining NaN with column mean
8. Standardize features (mean 0, SD 1) for scale-sensitive models

In [ ]:
# --------------------------------------------------------------
# 4a. Choose target IQ column (QIT, QIV, QIP, …)
# --------------------------------------------------------------

# TODO: pick one of the WPPSI-III IQ columns present after merge
TARGET_COLUMN = "QIT"  # options: "QIT" (Full-Scale), "QIV" (Verbal), "QIP" (Performance)


# --------------------------------------------------------------
# 4b. Filter to M0 (birth) passages
# --------------------------------------------------------------

df_linear_m0 = df_linear[df_linear["Passage"] == "M0"].copy()
df_nonlinear_m0 = df_nonlinear[df_nonlinear["Passage"] == "M0"].copy()

print("Linear M0 rows:", df_linear_m0.shape[0])
print("Nonlinear M0 rows:", df_nonlinear_m0.shape[0])


# --------------------------------------------------------------
# 4c. Drop WPPSI rows missing the chosen target
# --------------------------------------------------------------

df_wppsi_clean = df_wppsi.dropna(subset=[TARGET_COLUMN]).copy()
print(f"Children with valid {TARGET_COLUMN}:", df_wppsi_clean.shape[0])


# --------------------------------------------------------------
# 4d. Merge linear + nonlinear HRV on Monogramme + DDN + Passage
# --------------------------------------------------------------

hrv_merge_keys = ["Monogramme", "DDN", "Passage"]
df_hrv = df_linear_m0.merge(df_nonlinear_m0, on=hrv_merge_keys, how="inner", suffixes=("_lin", "_nonlin"))
print("Combined HRV shape:", df_hrv.shape)


# --------------------------------------------------------------
# 4e. Merge HRV + WPPSI on Monogramme + DDN
# --------------------------------------------------------------

wppsi_merge_keys = ["Monogramme", "DDN"]
df = df_hrv.merge(df_wppsi_clean, on=wppsi_merge_keys, how="inner")
print("Final dataset shape:", df.shape)


# --------------------------------------------------------------
# 4f. Drop fully-empty feature columns
# --------------------------------------------------------------

df = df.dropna(axis=1, how="all")
print("After dropping empty columns:", df.shape)


# --------------------------------------------------------------
# 4g. Separate features (X) and target (y)
# --------------------------------------------------------------

# Identifier / metadata columns that are not features
ID_COLS = ["Numéro", "Monogramme", "DDN", "Sexe", "Passage", "Date", "Recorder"]
ID_COLS = [c for c in ID_COLS if c in df.columns]

feature_cols = [c for c in df.columns if c not in ID_COLS + [TARGET_COLUMN]]

X = df[feature_cols].copy()
y = df[TARGET_COLUMN].copy()



## 5. Model Selection & Training

Choose a regression model. Because the sample is small (~189) relative to the
number of features (~100), prefer regularized models (Ridge, Lasso, Elastic Net,
Bayesian Ridge) and tune hyperparameters with nested cross-validation to avoid overfitting.

Uncomment the model you want to train below.

In [ ]:
# --------------------------------------------------------------
# TODO: Select and configure your regression model
# --------------------------------------------------------------

# Example models:
# model = Ridge(alpha=1.0, random_state=42)
# model = Lasso(alpha=0.1, random_state=42, max_iter=10000)
# model = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=10000)
# model = BayesianRidge()
# model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
# model = GradientBoostingRegressor(random_state=42)
# model = SVR(kernel="rbf")

# # Fit on scaled training data
# model.fit(X_train_scaled, y_train)

# print(f"Trained {model.__class__.__name__}")

print("Placeholder: uncomment and fit a model above.")

## 6. Model Evaluation

Per `dataset.md` §9, report **MAE** (Mean Absolute Error) and **R²**:
- **R² > 0** → model is better than predicting the mean
- **R² = 0** → same as the mean
- **R² < 0** → worse than guessing the mean

For the official cross-method comparison (same split, same preprocessing for all
groups), see `test_for_model.py` at the project root.

In [ ]:
# --------------------------------------------------------------
# TODO: Generate predictions and compute MAE + R²
# --------------------------------------------------------------

# y_pred = model.predict(X_test_scaled)

# mae = mean_absolute_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# print(f"MAE  : {mae:.4f}")
# print(f"R²   : {r2:.4f}")
# print(f"RMSE : {rmse:.4f}")

# # 5-fold cross-validated R² on training data
# cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="r2")
# print(f"\nCV R² (5-fold): {cv_r2.mean():.4f} (+/- {cv_r2.std():.4f})")

print("Placeholder: compute evaluation metrics above.")


## 7. Visualization

Predicted vs. actual QIT and residual plot to diagnose model performance.

In [ ]:
# --------------------------------------------------------------
# TODO: Uncomment and adapt for your trained model predictions
# --------------------------------------------------------------

# y_pred = model.predict(X_test_scaled)
# residuals = y_test - y_pred

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Predicted vs Actual
# axes[0].scatter(y_test, y_pred, alpha=0.6, edgecolors="k")
# axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
# axes[0].set_xlabel("Actual QIT")
# axes[0].set_ylabel("Predicted QIT")
# axes[0].set_title("Predicted vs Actual QIT")

# # Residuals
# axes[1].scatter(y_pred, residuals, alpha=0.6, edgecolors="k")
# axes[1].axhline(y=0, color="r", linestyle="--", lw=2)
# axes[1].set_xlabel("Predicted QIT")
# axes[1].set_ylabel("Residuals")
# axes[1].set_title("Residual Plot")

# plt.tight_layout()
# plt.show()

print("Placeholder: add visualization code above.")

## 8. Save / Load Model

Persist the trained model (and scaler) to disk for later inference.

In [ ]:
# --------------------------------------------------------------
# TODO: Save your trained model
# --------------------------------------------------------------

# joblib.dump(model, "regression_model.pkl")
# joblib.dump(scaler, "scaler.pkl")
# print("Model and scaler saved.")

# # To load later:
# loaded_model = joblib.load("regression_model.pkl")
# loaded_scaler = joblib.load("scaler.pkl")

print("Placeholder: save/load your model above.")

---
## Next Steps

1. Confirm the QIT column name matches `Résultats WPPSI III.csv` (case-sensitive).
2. Uncomment and fit a model in Section 5 (Ridge / Lasso / Elastic Net recommended for small n, high p).
3. Add hyperparameter tuning with nested cross-validation (inner CV for tuning, outer CV for performance).
4. Run `test_for_model.py` from the project root for the standardized cross-group comparison.
5. Report MAE and R² per `dataset.md` §9.